# Fase 6 — Kalman cm-space state estimation

Filtro de Kalman **explícito en centímetros**, encima de la homografía por líneas (nb07), como **capa de análisis post-asociación**. No re-hace detección/tracking/IDs.

**Tres aportes (mismo filtro):** (1) estado posición+velocidad principiado; (2) **puenteo de oclusión** (predict-only) — estima dónde está el objeto aunque no se vea; (3) velocidad físicamente suave + gating de outliers.

> El tracker (ByteTrack/BoT-SORT) ya trae un Kalman, pero en **píxeles** y solo para la asociación; su estado no se expone. Aquí lo exponemos en **cm**, calibrado al error de homografía.

## 1. Modelo de estado (velocidad constante, 2D)
Estado por objeto: $\mathbf{x}=[p_x,p_y,v_x,v_y]^\top$ (cm, cm/s).

**Predict** (cada frame, $dt$ real entre muestras):
$$\mathbf{x}^- = F(dt)\,\mathbf{x}, \qquad P^- = F P F^\top + Q(dt)$$
$$F(dt)=\begin{bmatrix}1&0&dt&0\\0&1&0&dt\\0&0&1&0\\0&0&0&1\end{bmatrix}, \quad Q(dt)=\sigma_a^2\begin{bmatrix}\tfrac{dt^4}{4}&0&\tfrac{dt^3}{2}&0\\0&\tfrac{dt^4}{4}&0&\tfrac{dt^3}{2}\\\tfrac{dt^3}{2}&0&dt^2&0\\0&\tfrac{dt^3}{2}&0&dt^2\end{bmatrix}$$

**Update** (si hay detección; $\mathbf{z}$ = posición en cm):
$$\mathbf{y}=\mathbf{z}-H\mathbf{x}^-,\;\; S=HP^-H^\top+R,\;\; K=P^-H^\top S^{-1},\;\; \mathbf{x}=\mathbf{x}^-+K\mathbf{y},\;\; P=(I-KH)P^-$$
con $H=\begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix}$ y $R=\sigma_z^2 I_2$ (calibrado del error de homografía, $\sigma_z=2$ cm).

**Oclusión** (sin detección → *predict-only*): $\mathbf{x}=\mathbf{x}^-,\,P=P^-$ → la trayectoria continúa y $\sigma_{pos}=\sqrt{P_{00}+P_{11}}$ **crece**.

**Gating** (reemplaza el corte duro de 300 cm/s): outlier si $d^2=\mathbf{y}^\top S^{-1}\mathbf{y} > \chi^2_2(0.99)=9.21$ → se salta el update, no tira el track.

In [ ]:
import sys; from pathlib import Path
REPO = Path('/workspace/FutBotMX-UAQTeam'); sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO/'notebooks/fase_6_kalman'))
from cm_positions_lines import compute_cm_positions_lines
from src.core.kalman_kinematics import compute_kalman_states
TRACKS = REPO/'outputs/inference/fase5_clips/IMG_9933_5m30/IMG_9933_5m30.json'

## 2. Posiciones en cm (homografía por líneas, nb07)
Arregla el *size-mismatch* de T3: usa `VideoHomographyLines` (de Rodrigo) + redimensiona el frame a la resolución de la máscara.

In [ ]:
cm = compute_cm_positions_lines(TRACKS)
print('posiciones cm:', len(cm.posiciones), '| fps:', cm.resumen.get('fps'))

## 3. Estados Kalman (cm, oclusiones rellenadas)

In [ ]:
kf = compute_kalman_states(cm)
print('objetos:', kf.resumen['n_obj'], '| oclusión rellenada:', kf.resumen['frames_rellenados_oclusion'], 'frames')
for o in kf.por_obj[:4]:
    print(f'  obj {o.obj_id} ({o.cls}): v_max {o.v_max_cms} cm/s, dist {o.dist_cm} cm, predichos {o.n_predicted}')

## 4. Ablación NIS + DESCUBRIMIENTO
`03_kalman_ablation.py` barre $\sigma_z,\sigma_a$ → NIS$\approx$2. Las innovaciones reales son **~2 cm** (ruido temporal de la homografía, no el sesgo absoluto 9–23 cm).

**Descubrimiento (balístico vs maniobra):**
- **Balón** (suave): Kalman **vence** a la extrapolación lineal en oclusión (2.65 vs 4.57 cm @ 12 frames). El balón es el objeto crítico → el KF aporta donde más importa.
- **Robot** (maniobra): ninguna extrapolación con velocidad vence a *hold* (KF 9.68 > linear 8.53 > hold 4.59). El modelo velocidad-constante no modela un robot que gira → motiva **CA/IMM**.

Experimentos: `01_kalman_experiment.py` (T6.1 oclusión, T6.2 suavidad, T6.5 NIS); tablas en `assets/fase6/tables/`.

In [ ]:
import csv
for f in ['T6_1_occlusion_recovery','T6_5_nis']:
    p = REPO/f'assets/fase6/tables/{f}.csv'
    if p.exists():
        print('==', f); [print(' ', r) for r in list(csv.reader(p.open()))[:7]]

## 5. Integración + demo (sin tocar el código del compañero)
El `MetricResult` Kalman se pasa a las funciones de eventos (posesión/zonas/gol geométrico) — **solo se llaman**, no se modifican (`02_kalman_integration.py`). El demo de 5 paneles (Original | Segmentación | Tracking | Minimap homografía+Kalman | Heatmap de él) lo arma `06_demo_kalman_minimap.py`.

**Para el análisis de eventos:** se trabaja **sobre** estos estados Kalman (más suaves, sin huecos), no sobre T3/T4 crudo.

In [ ]:
# render del demo (CPU, ~min). Descomenta para generar el mp4 de 5 paneles:
# %run 06_demo_kalman_minimap.py